In [1]:
from glob import glob
import re
import json
import os

#### Merge chunks into a single big json for now

In [2]:
ROOT_DIR = "/home/brandon/ill/pydata/"

PREVIOUS_PLOT_MAP = "../../../shared/asset_defs/gen/plot_map.json"
PREVIOUS_METADATA = "../../../shared/asset_defs/gen/plot_metadata.json"



In [3]:
# merge with previous plot map
output = {}
if os.path.exists(PREVIOUS_PLOT_MAP):
    with open(PREVIOUS_PLOT_MAP, "r") as f:
        output = json.load(f)
        pass

In [4]:
for path in glob(f"{ROOT_DIR}/alpha_world/plots/plot_map_*.txt"):
    chunk_name = re.match(r"^plot_map_(.+_.+).txt$", os.path.basename(path)).groups()[0]
    with open(path) as f:
        output[chunk_name] = f.read()
with open(f"{ROOT_DIR}/alpha_world/plots/plot_map.json", "w") as f:
    f.write(json.dumps(output))

# Metadata

In [5]:
metadata = {}
metadata_fields = set(["playerClaimable", "region"])
if os.path.exists(PREVIOUS_METADATA):
    with open(PREVIOUS_METADATA, "r") as f:
        for key, val in json.load(f).items():
            metadata[key] = {k: v for k, v in val.items() if k in metadata_fields}


In [6]:
seen_metadata = {}
generation_order = [(0, 0), (0, -1), (-1, 0), (-1, -1)]
for path in glob(f"{ROOT_DIR}/alpha_world/plots/plot_metadata_*.json"):
    chunk_name = re.match(r"^plot_metadata_(.+_.+).json$", os.path.basename(path)).groups()[0]
    with open(path, "r") as f:
        local_metadata = json.load(f)
        for key, val in local_metadata.items():
            filtered_metadata = {k: v for k, v in val.items() if k in metadata_fields}
            if key in seen_metadata:
                conflict = False
                for k, v in filtered_metadata.items():
                    if metadata[key][k] != v:
                        conflict = True
                        break
                if conflict:
                    print("Potential ID conflict: ", key, chunk_name, seen_metadata[key])
                    print(filtered_metadata, metadata[key])
            seen_metadata[key] = chunk_name
            metadata[key] = filtered_metadata
with open(f"{ROOT_DIR}/alpha_world/plots/plot_metadata.json", "w") as f:
    f.write(json.dumps(metadata))

In [7]:
spacing = 24
CHUNK_PADDING = spacing * 3
chunk_size = 2048
PADDING_MASK_ID = 5664385117739341

for [chunk_name, data] in output.items():
    for x in range(chunk_size):
        for z in range(chunk_size):
            json_index = voxeloo.runs.Index_U64()
            id = json_index.get(CHUNK_PADDING + x + (CHUNK_PADDING + z) * (chunk_size + CHUNK_PADDING * 2))
            if id == PADDING_MASK_ID:
                print(f"Padding mask found in {chunk_name} at {x}, {z}")

{'playerClaimable': False, 'region': 'roads'}
